#Code With Alpha Beta Pruning

In [ ]:
import math
import copy

class DotsAndBoxes:
    def __init__(self, size=2): # Reduced size for a playable example
        self.size = size
        self.num_dots = size + 1
        self.board = {}
        self.scores = {1: 0, 2: 0}
        self.current_player = 1

    # --- Game Mechanics ---

    def is_valid_move(self, r, c, orientation):
        if orientation not in ('H', 'V'):
            return False
        if orientation == 'H':
            is_in_bounds = (0 <= r < self.size) and (0 <= c < self.num_dots)
        else: # 'V'
            is_in_bounds = (0 <= r < self.num_dots) and (0 <= c < self.size)
        return is_in_bounds and (r, c, orientation) not in self.board

    def make_move(self, r, c, orientation):
        if not self.is_valid_move(r, c, orientation):
            return False, 0

        self.board[(r, c, orientation)] = self.current_player
        boxes_scored = self._check_new_boxes(r, c, orientation)

        if boxes_scored > 0:
            self.scores[self.current_player] += boxes_scored
            return True, boxes_scored
        else:
            self.current_player = 3 - self.current_player
            return True, 0

    def _check_new_boxes(self, r, c, orientation):
        boxes_completed = 0

        if orientation == 'H':
            # Check box below
            if r < self.size and c < self.size:
                if all(line in self.board for line in [(r + 1, c, 'H'), (r, c, 'V'), (r, c + 1, 'V')]):
                    boxes_completed += 1
            # Check box above
            if r > 0 and c < self.size:
                if all(line in self.board for line in [(r - 1, c, 'H'), (r, c, 'V'), (r, c + 1, 'V')]):
                    boxes_completed += 1

        else: # orientation == 'V'
            # Check box right
            if r < self.size and c < self.size:
                if all(line in self.board for line in [(r, c + 1, 'V'), (r, c, 'H'), (r + 1, c, 'H')]):
                    boxes_completed += 1
            # Check box left
            if r < self.size and c > 0:
                if all(line in self.board for line in [(r, c - 1, 'V'), (r, c - 1, 'H'), (r + 1, c - 1, 'H')]):
                    boxes_completed += 1

        return boxes_completed

    def get_available_moves(self):
        moves = []
        for r in range(self.size):
            for c in range(self.num_dots):
                if (r, c, 'H') not in self.board:
                    moves.append((r, c, 'H'))
        for r in range(self.num_dots):
            for c in range(self.size):
                if (r, c, 'V') not in self.board:
                    moves.append((r, c, 'V'))
        return moves

    def is_game_over(self):
        total_boxes = self.size * self.size
        return (self.scores[1] + self.scores[2]) == total_boxes

    def _clone_game(self):
        new_game = DotsAndBoxes(self.size)
        new_game.board = self.board.copy()
        new_game.scores = self.scores.copy()
        new_game.current_player = self.current_player
        return new_game

    # --- MINIMAX AI Implementation ---
    def minimax_evaluation(self, depth, alpha, beta, is_maximizing_player):
        """Minimax with Alpha-Beta Pruning, returns only the evaluation score."""
        if depth == 0 or self.is_game_over():
            return self.scores[1] - self.scores[2]

        if is_maximizing_player: # Player 1 (Max)
            max_eval = -math.inf
            for move in self.get_available_moves():
                temp_game = self._clone_game()
                _, boxes_scored = temp_game.make_move(*move)

                next_is_maximizing = True if boxes_scored > 0 else False
                evaluation = temp_game.minimax_evaluation(depth - 1, alpha, beta, next_is_maximizing)

                max_eval = max(max_eval, evaluation)
                alpha = max(alpha, max_eval)
                if beta <= alpha:
                    break
            return max_eval

        else: # Player 2 (Min)
            min_eval = math.inf
            for move in self.get_available_moves():
                temp_game = self._clone_game()
                _, boxes_scored = temp_game.make_move(*move)

                next_is_maximizing = False if boxes_scored > 0 else True
                evaluation = temp_game.minimax_evaluation(depth - 1, alpha, beta, next_is_maximizing)

                min_eval = min(min_eval, evaluation)
                beta = min(beta, min_eval)
                if beta <= alpha:
                    break
            return min_eval

    def find_best_move(self, depth):
        """
        Finds the best move for the current player using Minimax with Alpha-Beta pruning.
        Returns the best move (r, c, orientation).
        """
        if self.is_game_over():
            return None

        if self.current_player == 1:
            best_score, best_move = self._minimax_decision(depth, -math.inf, math.inf, True)
        else:
            best_score, best_move = self._minimax_decision(depth, -math.inf, math.inf, False)

        # Fallback to a random move if search depth is 0 or all moves are equally bad
        if best_move is None and self.get_available_moves():
            return self.get_available_moves()[0] # Return the first available move as fallback

        return best_move

    def _minimax_decision(self, depth, alpha, beta, is_maximizing_player):
        """Modified minimax to return (evaluation, best_move) for the top-level call."""
        available_moves = self.get_available_moves()

        if is_maximizing_player: # Player 1 (Max)
            max_eval = -math.inf
            best_move = None
            for move in available_moves:
                temp_game = self._clone_game()
                _, boxes_scored = temp_game.make_move(*move)

                next_is_maximizing = True if boxes_scored > 0 else False
                evaluation = temp_game.minimax_evaluation(depth - 1, alpha, beta, next_is_maximizing)

                if evaluation > max_eval:
                    max_eval = evaluation
                    best_move = move

                alpha = max(alpha, max_eval)
                if beta <= alpha:
                    break
            return max_eval, best_move

        else: # Player 2 (Min)
            min_eval = math.inf
            best_move = None
            for move in available_moves:
                temp_game = self._clone_game()
                _, boxes_scored = temp_game.make_move(*move)

                next_is_maximizing = False if boxes_scored > 0 else True
                evaluation = temp_game.minimax_evaluation(depth - 1, alpha, beta, next_is_maximizing)

                if evaluation < min_eval:
                    min_eval = evaluation
                    best_move = move

                beta = min(beta, min_eval)
                if beta <= alpha:
                    break
            return min_eval, best_move

# --- Helper Function for Visualization ---
def print_board(game: DotsAndBoxes):
    """Prints the current state of the Dots and Boxes board."""
    size = game.size
    print("\n  ", end="")
    for i in range(size + 1):
        print(f" {i} ", end="")
    print()

    # Draw horizontal lines and boxes
    for r in range(size):
        # Draw Dots and Horizontal Lines
        print(f"{r} ", end="")
        for c in range(size):
            line = '---' if (r, c, 'H') in game.board else '   '
            print(f"•{line}", end="")
        print("•") # End dot of the row

        # Draw Vertical Lines and Box Owners
        print("  ", end="")
        for c in range(size + 1):
            v_line = '|' if (r, c, 'V') in game.board else ' '
            box_owner = ' '
            if c < size:
                # Check box ownership
                # A box at (r, c) is formed by (r, c, H), (r+1, c, H), (r, c, V), (r, c+1, V)
                if all(l in game.board for l in [(r, c, 'H'), (r + 1, c, 'H'), (r, c, 'V'), (r, c + 1, 'V')]):
                    # Check which player scored it (a simpler proxy for ownership)
                    # This check is an approximation; a true game tracks box ownership separately.
                    if game.board.get((r,c,'H')) == game.board.get((r+1,c,'H')) == game.board.get((r,c,'V')) == game.board.get((r,c+1,'V')):
                         box_owner = str(game.board.get((r,c,'H')))
                    else: # If ownership is mixed, just display an 'X' or 'B' (Box)
                        box_owner = 'B'

            print(f"{v_line} {box_owner} ", end="")
        print()

    # Draw final row of dots
    print(f"{size} ", end="")
    for c in range(size):
        line = '---' if (size, c, 'H') in game.board else '   '
        print(f"•{line}", end="")
    print("•")


# --- Main Game Loop Implementation ---
def main():
    """Implements a game loop for Human (P1) vs. AI (P2)."""
    # Use a 2x2 board (size=2) for fast AI search (4 total boxes)
    game = DotsAndBoxes(size=2)

    # Set AI search depth.
    # Max depth for a 2x2 board is 24 moves, so a small depth is necessary for quick play.
    AI_DEPTH = 5

    print("Welcome to Dots and Boxes (2x2 Board)!")
    print(f"Player 1: Human (Input: r, c, H/V)")
    print(f"Player 2: AI (Minimax Depth {AI_DEPTH})")

    while not game.is_game_over():
        print("\n" + "="*30)
        print(f"Current Player: {game.current_player}")
        print(f"Scores: P1={game.scores[1]}, P2={game.scores[2]}")
        print_board(game)

        player = game.current_player
        boxes_scored = 0

        if player == 1:
            # Human Player (P1)
            print(f"\nPlayer 1's turn (r: 0-{game.size-1} H, 0-{game.size} V; c: 0-{game.num_dots-1} H, 0-{game.size-1} V):")
            while True:
                try:
                    move_str = input("Enter move (e.g., '0,1,H' or '1,0,V'): ").strip().split(',')
                    if len(move_str) != 3:
                        raise ValueError

                    r = int(move_str[0])
                    c = int(move_str[1])
                    orientation = move_str[2].upper()

                    success, boxes_scored = game.make_move(r, c, orientation)
                    if success:
                        print(f"P1 scored {boxes_scored} box(es).")
                        break
                    else:
                        print("Invalid or already played line. Try again.")
                except Exception as e:
                    print("Invalid input format. Please use r,c,H/V (e.g., 0,0,H).")

        else:
            # AI Player (P2)
            print("\nPlayer 2 (AI) is thinking...")
            best_move = game.find_best_move(AI_DEPTH)

            if best_move:
                r, c, orientation = best_move
                print(f"AI plays: {r}, {c}, {orientation}")

                success, boxes_scored = game.make_move(r, c, orientation)
                if success:
                    print(f"P2 scored {boxes_scored} box(es).")

                # The AI should always make a valid move if it finds one, so no error check here.
            else:
                print("AI could not find a move. Game should be over or there's an error.")
                break

        # A player's turn continues if they scored a box
        if boxes_scored > 0:
            print(f"Player {3-game.current_player}'s turn continues!")

    # --- Game Over ---
    print("\n" + "#"*30)
    print("               GAME OVER               ")
    print("#"*30)
    print(f"Final Scores: Player 1: {game.scores[1]}, Player 2: {game.scores[2]}")

    if game.scores[1] > game.scores[2]:
        print("🎉 Player 1 (Human) wins!")
    elif game.scores[2] > game.scores[1]:
        print("🤖 Player 2 (AI) wins!")
    else:
        print("🤝 It's a draw!")
    print_board(game)


if __name__ == '__main__':
    main()

Welcome to Dots and Boxes (2x2 Board)!
Player 1: Human (Input: r, c, H/V)
Player 2: AI (Minimax Depth 5)

Current Player: 1
Scores: P1=0, P2=0

   0  1  2 
0 •   •   •
              
1 •   •   •
              
2 •   •   •

Player 1's turn (r: 0-1 H, 0-2 V; c: 0-2 H, 0-1 V):
Enter move (e.g., '0,1,H' or '1,0,V'): 0,0,H
P1 scored 0 box(es).

Current Player: 2
Scores: P1=0, P2=0

   0  1  2 
0 •---•   •
              
1 •   •   •
              
2 •   •   •

Player 2 (AI) is thinking...
AI plays: 0, 1, H
P2 scored 0 box(es).

Current Player: 1
Scores: P1=0, P2=0

   0  1  2 
0 •---•---•
              
1 •   •   •
              
2 •   •   •

Player 1's turn (r: 0-1 H, 0-2 V; c: 0-2 H, 0-1 V):
Enter move (e.g., '0,1,H' or '1,0,V'): 0,1,V
P1 scored 0 box(es).

Current Player: 2
Scores: P1=0, P2=0

   0  1  2 
0 •---•---•
      |       
1 •   •   •
              
2 •   •   •

Player 2 (AI) is thinking...
AI plays: 0, 2, H
P2 scored 0 box(es).

Current Player: 1
Scores: P1=0, P2=0

   0  1  2 


#Code With Min-Max Algorithm

In [ ]:
import math
import copy

class DotsAndBoxes:
    def __init__(self, size=3):
        # A 3x3 game has 4x4 dots (size+1) and a 3x3 grid of boxes (size x size)
        self.size = size
        self.num_dots = size + 1
        # The board state is a dictionary: (r, c, orientation) -> player_id (1 or 2)
        self.board = {}
        self.scores = {1: 0, 2: 0}
        self.current_player = 1

    # --- Game Mechanics ---

    def is_valid_move(self, r, c, orientation):
        """Checks if a line can be drawn."""
        if orientation not in ('H', 'V'):
            return False
        if orientation == 'H':
            is_in_bounds = (0 <= r < self.size) and (0 <= c < self.num_dots)
        else: # 'V'
            is_in_bounds = (0 <= r < self.num_dots) and (0 <= c < self.size)

        return is_in_bounds and (r, c, orientation) not in self.board

    def make_move(self, r, c, orientation):
        """Draws a line and checks for completed boxes."""
        if not self.is_valid_move(r, c, orientation):
            return False, 0

        self.board[(r, c, orientation)] = self.current_player
        boxes_scored = self._check_new_boxes(r, c, orientation)

        if boxes_scored > 0:
            self.scores[self.current_player] += boxes_scored
            # Player gets another turn (current_player remains the same)
            return True, boxes_scored
        else:
            # Switch player
            self.current_player = 3 - self.current_player
            return True, 0

    def _check_new_boxes(self, r, c, orientation):
        """Checks if the new line completes any boxes."""
        boxes_completed = 0

        if orientation == 'H':
            # Check box below (starts at row r, column c)
            if r < self.size and c < self.size:
                if all(line in self.board for line in [(r + 1, c, 'H'), (r, c, 'V'), (r, c + 1, 'V')]):
                    boxes_completed += 1
            # Check box above (starts at row r-1, column c)
            if r > 0 and c < self.size:
                if all(line in self.board for line in [(r - 1, c, 'H'), (r, c, 'V'), (r, c + 1, 'V')]):
                    boxes_completed += 1

        else: # orientation == 'V'
            # Check box right (starts at row r, column c)
            if r < self.size and c < self.size:
                if all(line in self.board for line in [(r, c + 1, 'V'), (r, c, 'H'), (r + 1, c, 'H')]):
                    boxes_completed += 1
            # Check box left (starts at row r, column c-1)
            if r < self.size and c > 0:
                if all(line in self.board for line in [(r, c - 1, 'V'), (r, c - 1, 'H'), (r + 1, c - 1, 'H')]):
                    boxes_completed += 1

        return boxes_completed

    def get_available_moves(self):
        """Generates all possible moves on the current board."""
        moves = []
        # Horizontal moves
        for r in range(self.size):
            for c in range(self.num_dots):
                if (r, c, 'H') not in self.board:
                    moves.append((r, c, 'H'))
        # Vertical moves
        for r in range(self.num_dots):
            for c in range(self.size):
                if (r, c, 'V') not in self.board:
                    moves.append((r, c, 'V'))
        return moves

    def is_game_over(self):
        """The game ends when all possible lines are drawn."""
        total_lines = 2 * self.size * self.num_dots
        return len(self.board) == total_lines

    def _clone_game(self):
        """Creates a deep copy of the current game state."""
        new_game = DotsAndBoxes(self.size)
        new_game.board = self.board.copy()
        new_game.scores = self.scores.copy()
        new_game.current_player = self.current_player
        return new_game

    # --- PURE MINIMAX ALGORITHM ---

    # We keep alpha and beta in the signature for consistency, but ignore them in the logic
    def minimax(self, depth, alpha, beta, is_maximizing_player):
        """
        The pure Minimax algorithm (no pruning).
        Returns the score from the perspective of the original Max player (Player 1).
        """
        if depth == 0 or self.is_game_over():
            # Evaluation: Player 1's score - Player 2's score
            return self.scores[1] - self.scores[2]

        if is_maximizing_player: # Player 1 (Max)
            max_eval = -math.inf
            for move in self.get_available_moves():
                temp_game = self._clone_game()
                _, boxes_scored = temp_game.make_move(*move)

                # Determine next call state based on scoring an extra turn
                next_is_maximizing = True if boxes_scored > 0 else False

                # Recursive call, ignoring alpha/beta
                evaluation = temp_game.minimax(depth - 1, -math.inf, math.inf, next_is_maximizing)

                max_eval = max(max_eval, evaluation)

            return max_eval

        else: # Player 2 (Min)
            min_eval = math.inf
            for move in self.get_available_moves():
                temp_game = self._clone_game()
                _, boxes_scored = temp_game.make_move(*move)

                # Determine next call state based on scoring an extra turn
                next_is_maximizing = False if boxes_scored > 0 else True

                # Recursive call, ignoring alpha/beta
                evaluation = temp_game.minimax(depth - 1, -math.inf, math.inf, next_is_maximizing)

                min_eval = min(min_eval, evaluation)

            return min_eval

    def find_best_move(self, depth=4):
        """Public method to find the optimal move for the current player using Minimax."""
        best_move = None
        is_maximizing_player = (self.current_player == 1)

        # We define a wide alpha/beta range, but it's not used for pruning
        alpha = -math.inf
        beta = math.inf

        if is_maximizing_player:
            best_score = -math.inf

            for move in self.get_available_moves():
                temp_game = self._clone_game()
                _, boxes_scored = temp_game.make_move(*move)

                next_is_maximizing = True if boxes_scored > 0 else False

                score = temp_game.minimax(depth - 1, alpha, beta, next_is_maximizing)

                if score > best_score:
                    best_score = score
                    best_move = move

        else: # Current player is 2 (Minimizing for P1's score)
            best_score = math.inf

            for move in self.get_available_moves():
                temp_game = self._clone_game()
                _, boxes_scored = temp_game.make_move(*move)

                next_is_maximizing = False if boxes_scored > 0 else True

                score = temp_game.minimax(depth - 1, alpha, beta, next_is_maximizing)

                if score < best_score:
                    best_score = score
                    best_move = move

        return best_move, best_score

    # --- Utility/Display ---

    def display_board(self):
        """Prints the current state of the board."""
        # Helper to check if a box is closed
        def is_box_closed(r, c):
            return all(line in self.board for line in [
                (r, c, 'H'), (r + 1, c, 'H'), (r, c, 'V'), (r, c + 1, 'V')
            ])

        # Prints a simplified ownership for the display function
        def get_box_char(r, c):
            # Note: This is an incomplete/simplified box-owner check for display purposes,
            # relying on the game state to find a player who closed it.
            if is_box_closed(r, c):
                # Search for the *last* line played to close the box
                all_lines = [(r, c, 'H'), (r + 1, c, 'H'), (r, c, 'V'), (r, c + 1, 'V')]
                # Find the player of the line played last (simplistic tie-breaker)
                if all_lines[-1] in self.board:
                    player = self.board.get(all_lines[0])
                    return 'X' if player == 1 else 'O'
            return ' '

        print("\n   " + "—" * (self.size * 4 + 1))

        for r in range(self.size):
            # Line 1: Dots and Horizontal Lines
            h_line_str = ""
            for c in range(self.size):
                h_line = '—' if (r, c, 'H') in self.board else ' '
                h_line_str += f" {h_line}{h_line}{h_line}*"
            print(f"{r}:*" + h_line_str)

            # Line 2: Vertical Lines and Boxes
            box_str = ""
            for c in range(self.size):
                v_line_left = '|' if (r, c, 'V') in self.board else ' '
                box_char = get_box_char(r, c)
                box_str += f"{v_line_left} {box_char} "

            v_line_right = '|' if (r, self.size, 'V') in self.board else ' '
            print(f"  {box_str}{v_line_right}")

        # Print bottom horizontal line (last row of horizontal lines)
        h_line_str = ""
        for c in range(self.size):
            h_line = '—' if (self.size, c, 'H') in self.board else ' '
            h_line_str += f" {h_line}{h_line}{h_line}*"
        print(f"{self.size}:*" + h_line_str)

        print("   " + "—" * (self.size * 4 + 1))
        print(f"Player 1 (X) Score: {self.scores[1]}, Player 2 (O) Score: {self.scores[2]}")
        print(f"Current Player: {'X' if self.current_player == 1 else 'O'}")


# --- Main Function / Game Loop ---

if __name__ == "__main__":
    # Use a small 2x2 grid (size=2) for Minimax to run in a reasonable time
    # A 3x3 grid is often too large for unoptimized Minimax.
    game = DotsAndBoxes(size=2)

    print("--- Dots and Boxes (2x2 Grid) - Pure Minimax AI ---")
    print("Player 1 (Human: X) vs. Player 2 (AI: O)")
    print("Moves are (r, c, orientation: H/V). e.g., 0,0,H")

    while not game.is_game_over():
        game.display_board()

        if game.current_player == 1:
            # Human player 1's turn
            print("Player 1 (Human: X) turn.")
            try:
                move_input = input("Enter move (r,c,o): ").upper().split(',')
                if len(move_input) != 3:
                    print("Invalid input format. Must be r,c,o.")
                    continue
                r, c, o = int(move_input[0]), int(move_input[1]), move_input[2].strip()

                valid, boxes_scored = game.make_move(r, c, o)

                if not valid:
                    print("Invalid or already drawn line. Try again.")
                    continue
                if boxes_scored > 0:
                    print(f"Player 1 scored {boxes_scored} box(es) and gets another turn!")
            except ValueError:
                print("Invalid number or format. Try again.")
                continue
            except Exception as e:
                print(f"An error occurred: {e}")
                continue

        else:
            # Pure Minimax AI player 2's turn
            print("Player 2 (AI: O) is thinking... (Please wait)")

            # Using a relatively deep search for the small 2x2 board
            move, score = game.find_best_move(depth=7)

            if move:
                r, c, o = move
                # Score is predicted P1 advantage (P1-P2)
                print(f"AI Move: ({r},{c},{o}). AI Predicted P1-P2 Score Difference: {score}")

                _, boxes_scored = game.make_move(r, c, o)
                if boxes_scored > 0:
                    print(f"Player 2 scored {boxes_scored} box(es) and gets another turn!")
            else:
                print("AI found no moves (Game should be over).")
                break

    # Final score
    print("\n— — — Game Over — — —")
    game.display_board()
    if game.scores[1] > game.scores[2]:
        print("Player 1 (X) Wins! 🎉")
    elif game.scores[2] > game.scores[1]:
        print("Player 2 (O/AI) Wins! 🤖")
    else:
        print("It's a Draw! 🤝")

--- Dots and Boxes (2x2 Grid) - Pure Minimax AI ---
Player 1 (Human: X) vs. Player 2 (AI: O)
Moves are (r, c, orientation: H/V). e.g., 0,0,H

   —————————
0:*    *    *
           
1:*    *    *
           
2:*    *    *
   —————————
Player 1 (X) Score: 0, Player 2 (O) Score: 0
Current Player: X
Player 1 (Human: X) turn.
Invalid or already drawn line. Try again.

   —————————
0:*    *    *
           
1:*    *    *
           
2:*    *    *
   —————————
Player 1 (X) Score: 0, Player 2 (O) Score: 0
Current Player: X
Player 1 (Human: X) turn.

   —————————
0:*    * ———*
           
1:*    *    *
           
2:*    *    *
   —————————
Player 1 (X) Score: 0, Player 2 (O) Score: 0
Current Player: O
Player 2 (AI: O) is thinking... (Please wait)
AI Move: (0,0,H). AI Predicted P1-P2 Score Difference: 0

   —————————
0:* ———* ———*
           
1:*    *    *
           
2:*    *    *
   —————————
Player 1 (X) Score: 0, Player 2 (O) Score: 0
Current Player: X
Player 1 (Human: X) turn.

   ————————